# SHAP-Style Plots Testing Notebook

This notebook is for manual testing of SHAP-style beeswarm plots.
It uses functions from `postprocess_functions.py` and `plot_shap_functions.py`.

Note: These are SHAP-*style* plots computed directly from data without requiring a trained model.

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import sys

# Add parent directory to path for imports
sys.path.insert(0, str(Path().resolve().parent))

from ddstartup.postprocessing.postprocess_functions import (
    load_h5_to_dataframe,
    get_input_parameters,
    scale_target,
    apply_filters,
    find_latest_h5_file
)
from ddstartup.postprocessing.plot_shap_functions import (
    compute_feature_importance,
    normalize_to_range,
    create_shap_style_beeswarm_plot
)
from ddstartup.utils.tools import PARAM_UNITS

print("✅ Imports successful")

## Configuration

In [ ]:
# Configuration - specify directory or files

# Option 1: Automatic - find latest file in specified directory (DEFAULT)
outputs_dir = Path('../outputs')
files_to_analyze = []

latest_h5_file = find_latest_h5_file(outputs_dir)
if latest_h5_file:
    print(f"📂 Auto-detected folder: {latest_h5_file.parent.name}")
    print(f"📄 Latest file: {latest_h5_file.name}")
    files_to_analyze = [latest_h5_file]

# Option 2: Analyze all files in a specific folder
# outputs_dir = Path('../outputs/20251008_081422_parametric_T_seeded')
# files_to_analyze = sorted(outputs_dir.glob('*.h5'))
# print(f"📂 Using folder: {outputs_dir.name}")
# print(f"📄 Found {len(files_to_analyze)} file(s): {[f.name for f in files_to_analyze]}")

# Target variable to analyze
target = 'Q_fusion'  # Change this to your desired target variable
target_unit = 'W'  # Specify the unit for the target
print(f"\n🎯 Target variable: {target} ({target_unit})")

## Load and Prepare Data

In [ ]:
# Load data
if not files_to_analyze:
    raise ValueError("No files to analyze. Please check the configuration.")

df = load_h5_to_dataframe(files_to_analyze)
print(f"📊 Loaded dataframe with shape: {df.shape}")
print(f"📋 Columns: {list(df.columns)}")

# Get input parameters
inputs = get_input_parameters(df)
print(f"\n🔧 Input parameters ({len(inputs)}): {inputs}")

# Check if target exists
if target not in df.columns:
    print(f"❌ Target '{target}' not found in dataframe")
    print(f"Available columns: {list(df.columns)}")
    raise ValueError(f"Target '{target}' not in dataframe")

print(f"\n📈 Target '{target}' range: [{df[target].min():.2e}, {df[target].max():.2e}]")
print(f"📊 Target statistics:")
print(df[target].describe())

## Apply Filters (Optional)

In [ ]:
# Optional: Apply filters
filters = {
    # Example: 'Q_fusion': {'min': 0},  # Only positive Q_fusion
    # Example: 'Ti_0': {'min': 5e3, 'max': 20e3},  # Temperature range
}

if filters:
    df_filtered = apply_filters(df, filters)
    print(f"🔍 Applied filters: {filters}")
    print(f"📊 Filtered dataframe shape: {df_filtered.shape} (was {df.shape})")
    df = df_filtered
else:
    print("No filters applied")

## Test 1: Compute Feature Importance

In [ ]:
# Compute feature importance using correlation
if len(inputs) > 0:
    print(f"Computing feature importance (correlation with {target})...\n")
    
    importance = compute_feature_importance(df, inputs, target)
    
    # Create a DataFrame for better display
    importance_df = pd.DataFrame({
        'Parameter': inputs,
        'Importance': importance
    }).sort_values('Importance', ascending=False)
    
    print("Feature Importance Ranking:")
    print("="*50)
    for idx, row in importance_df.iterrows():
        print(f"{row['Parameter']:20s} : {row['Importance']:.4f}")
    print("="*50)
    
    # Plot bar chart of importance
    plt.figure(figsize=(10, max(6, len(inputs)*0.3)))
    plt.barh(importance_df['Parameter'], importance_df['Importance'])
    plt.xlabel('Importance (|correlation|)')
    plt.title(f'Feature Importance for {target}')
    plt.tight_layout()
    plt.show()
else:
    print("❌ No input parameters available")

## Test 2: Check for Constant Parameters

In [ ]:
# Identify parameters with zero variance (constant values)
if len(inputs) > 0:
    print("Checking for constant parameters (zero variance)...\n")
    
    constant_params = []
    varying_params = []
    
    for param in inputs:
        if df[param].std() == 0:
            constant_params.append(param)
            print(f"❌ {param}: CONSTANT (value = {df[param].iloc[0]:.3e})")
        else:
            varying_params.append(param)
            print(f"✅ {param}: varying (std = {df[param].std():.3e})")
    
    print(f"\nSummary:")
    print(f"  Varying parameters: {len(varying_params)}")
    print(f"  Constant parameters: {len(constant_params)}")
    
    if constant_params:
        print(f"\n⚠️ Note: Constant parameters will be excluded from SHAP plots")
else:
    print("❌ No input parameters available")

## Test 3: Create SHAP-Style Beeswarm Plot (Full)

In [ ]:
# Create the full SHAP-style beeswarm plot
if len(inputs) > 0:
    # Create output directory for test plots
    test_outputs_dir = Path('../outputs/manual_test_shap')
    test_outputs_dir.mkdir(parents=True, exist_ok=True)
    
    print(f"Creating SHAP-style beeswarm plot for '{target}'...\n")
    
    result = create_shap_style_beeswarm_plot(
        df,
        input_parameters=inputs,
        target=target,
        target_unit=target_unit,
        outputs_dir=test_outputs_dir,
        plot_name=f'shap_beeswarm_{target}',
        max_display=20,  # Show top 20 features
        max_samples=2000  # Limit samples for performance
    )
    
    if result:
        print(f"\n✅ Plot saved to {test_outputs_dir}")
        print(f"   File: shap_beeswarm_{target}.png")
        
        # Display the saved image
        from IPython.display import Image, display
        display(Image(filename=test_outputs_dir / f'shap_beeswarm_{target}.png'))
    else:
        print("⚠️ Plot creation skipped (no varying parameters)")
else:
    print("❌ No input parameters available")

## Test 4: SHAP Plot with Limited Display

In [ ]:
# Create SHAP plot showing only top N most important features
if len(inputs) > 5:
    max_display = 10  # Show only top 10 features
    
    print(f"Creating SHAP-style plot with max_display={max_display}...\n")
    
    result = create_shap_style_beeswarm_plot(
        df,
        input_parameters=inputs,
        target=target,
        target_unit=target_unit,
        outputs_dir=test_outputs_dir,
        plot_name=f'shap_beeswarm_{target}_top{max_display}',
        max_display=max_display,
        max_samples=2000
    )
    
    if result:
        from IPython.display import Image, display
        display(Image(filename=test_outputs_dir / f'shap_beeswarm_{target}_top{max_display}.png'))
        print(f"\n✅ Top-{max_display} plot saved")
else:
    print("ℹ️ Skipping limited display test (need more than 5 inputs)")

## Test 5: SHAP Plot with Sample Limiting

In [ ]:
# Test with different sample limits to see performance impact
if len(inputs) > 0 and len(df) > 1000:
    max_samples = 500  # Limit to 500 samples for faster rendering
    
    print(f"Creating SHAP-style plot with max_samples={max_samples}...")
    print(f"(Total available samples: {len(df)})\n")
    
    result = create_shap_style_beeswarm_plot(
        df,
        input_parameters=inputs,
        target=target,
        target_unit=target_unit,
        outputs_dir=test_outputs_dir,
        plot_name=f'shap_beeswarm_{target}_limited',
        max_display=15,
        max_samples=max_samples
    )
    
    if result:
        from IPython.display import Image, display
        display(Image(filename=test_outputs_dir / f'shap_beeswarm_{target}_limited.png'))
        print(f"\n✅ Sample-limited plot saved")
else:
    print("ℹ️ Skipping sample limiting test (dataset too small)")

## Test 6: Analyze Multiple Targets

In [ ]:
# Create SHAP plots for multiple output variables
output_columns = [col for col in df.columns if col not in inputs and col != 'index']
targets_to_test = output_columns[:3] if len(output_columns) >= 3 else output_columns

if len(targets_to_test) > 1:
    print(f"Creating SHAP plots for multiple targets: {targets_to_test}\n")
    
    for tgt in targets_to_test:
        print(f"\n{'='*60}")
        print(f"Target: {tgt}")
        print(f"{'='*60}")
        
        # Get unit if available
        tgt_unit = PARAM_UNITS.get(tgt, '')
        
        result = create_shap_style_beeswarm_plot(
            df,
            input_parameters=inputs,
            target=tgt,
            target_unit=tgt_unit,
            outputs_dir=test_outputs_dir,
            plot_name=f'shap_beeswarm_{tgt}',
            max_display=10,
            max_samples=1000
        )
        
        if result:
            # Show top 3 most important features
            print(f"\nTop 3 most important features for {tgt}:")
            importance = compute_feature_importance(df, inputs, tgt)
            top_indices = np.argsort(importance)[::-1][:3]
            for i, idx in enumerate(top_indices, 1):
                print(f"  {i}. {inputs[idx]}: importance = {importance[idx]:.4f}")
    
    print(f"\n✅ All SHAP plots saved to {test_outputs_dir}")
else:
    print("ℹ️ Not enough output variables for multi-target test")

## Test 7: Manual Correlation Analysis

In [ ]:
# Verify correlation-based importance calculation manually
if len(inputs) > 0:
    print("Manual correlation verification:")
    print("="*70)
    
    # Compute correlations manually
    correlations = []
    for param in inputs[:5]:  # Show first 5 for brevity
        if df[param].std() > 0:
            corr = np.corrcoef(df[param].values, df[target].values)[0, 1]
            correlations.append((param, corr, abs(corr)))
            print(f"{param:20s} : corr = {corr:+.4f}, |corr| = {abs(corr):.4f}")
        else:
            print(f"{param:20s} : CONSTANT (no correlation)")
    
    print("="*70)
    
    if correlations:
        # Sort by absolute correlation
        correlations.sort(key=lambda x: x[2], reverse=True)
        print(f"\nMost correlated: {correlations[0][0]} (|r|={correlations[0][2]:.4f})")
        print(f"Least correlated: {correlations[-1][0]} (|r|={correlations[-1][2]:.4f})")

## Summary

### SHAP-Style Plot Functions Tested:

1. ✅ **compute_feature_importance**
   - Calculates importance using correlation with target
   - Returns absolute correlation values
   - Handles constant parameters (zero variance)

2. ✅ **normalize_to_range**
   - Normalizes values to [0, 1] for color mapping
   - Used internally for feature coloring

3. ✅ **create_shap_style_beeswarm_plot**
   - Creates SHAP-style visualization without ML model
   - Shows feature importance and value distribution
   - Color-coded by feature value (blue=low, red=high)
   - Horizontal position shows impact magnitude

### Key Features:

**Importance Calculation:**
- Based on correlation with target (no ML model needed)
- Absolute correlation = importance score
- Constant features automatically excluded

**Visualization:**
- Features ordered by importance (top to bottom)
- Each dot represents one sample
- Dot color indicates feature value:
  - Blue = low values
  - Red = high values
- Horizontal spread shows impact:
  - Correlation × standardized(value)
  - Standardized for fair comparison across features

**Parameters:**
- `max_display`: Number of top features to show (default: 20)
- `max_samples`: Maximum samples to plot per feature (default: 2000)
  - Reduces rendering time for large datasets
  - Randomly samples if dataset is larger

### Interpretation Guide:

**Reading the Plot:**
1. Top features = most important (highest |correlation|)
2. Spread width = impact magnitude
3. Color = feature value (blue→red = low→high)
4. Horizontal position = correlation × normalized value

**Common Patterns:**
- Wide spread = strong influence on target
- Color gradient = feature value affects target direction
- Narrow spread = weak influence

### Output Files:
- PNG: High-resolution beeswarm plot (150 DPI)
- Includes title, axis labels, and colorbar
- Ready for publication or presentation